# Piper Plus CPU 推論ベンチマーク

Google Colab で Piper Plus の CPU 推論速度を計測し、他の TTS エンジンと比較します。

**計測内容:**
- DEFAULT SessionOptions vs OPTIMIZED (ort_utils) の比較
- 短文 / 中文 / 長文の RTF (Real-Time Factor)
- CPU 情報の記録

> RTF = 推論時間 / 音声長。小さいほど速い。1.0 = リアルタイム。

## 1. 環境セットアップ

In [4]:
import platform, os

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"CPU: {platform.processor()}")
print(f"Logical cores: {os.cpu_count()}")

# Colab CPU details
!cat /proc/cpuinfo | grep 'model name' | head -1
!cat /proc/cpuinfo | grep 'cpu cores' | head -1

Python: 3.12.13
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
CPU: x86_64
Logical cores: 2
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
cpu cores	: 1


In [5]:
!pip install -q onnxruntime numpy huggingface_hub

import onnxruntime
print(f"ONNX Runtime: {onnxruntime.__version__}")
print(f"Available providers: {onnxruntime.get_available_providers()}")

# --- piper_train.ort_utils.create_session_options() inline version ---
MAX_INTRA_THREADS = 4

def create_session_options(*, intra_op_threads=None, inter_op_threads=1):
    opts = onnxruntime.SessionOptions()
    opts.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.execution_mode = onnxruntime.ExecutionMode.ORT_SEQUENTIAL
    env_threads = os.environ.get("PIPER_INTRA_THREADS")
    if env_threads is not None:
        opts.intra_op_num_threads = int(env_threads)
    elif intra_op_threads is not None:
        opts.intra_op_num_threads = intra_op_threads
    else:
        try:
            logical_cores = len(os.sched_getaffinity(0))
        except (AttributeError, OSError):
            logical_cores = os.cpu_count() or 2
        opts.intra_op_num_threads = min(logical_cores // 2 or 1, MAX_INTRA_THREADS)
    opts.inter_op_num_threads = inter_op_threads
    opts.enable_cpu_mem_arena = True
    opts.enable_mem_pattern = True
    opts.enable_mem_reuse = True
    return opts

print("create_session_options() defined.")

ONNX Runtime: 1.24.4
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
create_session_options() defined.


## 2. モデルダウンロード

In [6]:
from huggingface_hub import hf_hub_download

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = hf_hub_download(
    repo_id="ayousanz/piper-plus-tsukuyomi-chan",
    filename="tsukuyomi-chan-6lang-fp16.onnx",
    local_dir=MODEL_DIR,
)
config_path = hf_hub_download(
    repo_id="ayousanz/piper-plus-tsukuyomi-chan",
    filename="config.json",
    local_dir=MODEL_DIR,
)

model_size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"Model: {model_path} ({model_size_mb:.1f} MB)")
print(f"Config: {config_path}")

tsukuyomi-chan-6lang-fp16.onnx:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Model: /content/models/tsukuyomi-chan-6lang-fp16.onnx (37.4 MB)
Config: /content/models/config.json


## 3. ベンチマーク実行

In [7]:
import json
import time
import numpy as np

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

sample_rate = config["audio"]["sample_rate"]
print(f"Sample rate: {sample_rate}")

test_cases = [
    ("short  (20 phonemes)",  [1] + [8, 0] * 10 + [2]),
    ("medium (100 phonemes)", [1] + [8, 0] * 50 + [2]),
    ("long   (300 phonemes)", [1] + [8, 0] * 150 + [2]),
]

WARMUP = 3
RUNS = 10

def benchmark(label, sess_options):
    session = onnxruntime.InferenceSession(
        model_path, sess_options=sess_options, providers=["CPUExecutionProvider"]
    )
    input_names = [inp.name for inp in session.get_inputs()]
    has_sid = "sid" in input_names
    has_lid = "lid" in input_names
    has_prosody = "prosody_features" in input_names

    results = {}
    for name, phoneme_ids in test_cases:
        seq_len = len(phoneme_ids)
        inputs = {
            "input": np.array([phoneme_ids], dtype=np.int64),
            "input_lengths": np.array([seq_len], dtype=np.int64),
            "scales": np.array([0.667, 1.0, 0.8], dtype=np.float32),
        }
        if has_sid:
            inputs["sid"] = np.array([0], dtype=np.int64)
        if has_lid:
            inputs["lid"] = np.array([0], dtype=np.int64)
        if has_prosody:
            inputs["prosody_features"] = np.zeros((1, seq_len, 3), dtype=np.int64)

        for _ in range(WARMUP):
            session.run(None, inputs)

        times = []
        for _ in range(RUNS):
            start = time.perf_counter()
            outputs = session.run(None, inputs)
            elapsed = time.perf_counter() - start
            times.append(elapsed)

        audio_len = outputs[0].shape[-1]
        audio_sec = audio_len / sample_rate
        avg_time = sum(times) / len(times)
        rtf = avg_time / audio_sec if audio_sec > 0 else 0
        results[name] = {"avg_ms": avg_time * 1000, "rtf": rtf, "audio_sec": audio_sec}

    return results

print(f"Warmup: {WARMUP} runs, Benchmark: {RUNS} runs")

Sample rate: 22050
Warmup: 3 runs, Benchmark: 10 runs


In [8]:
print("=" * 60)
print("DEFAULT SessionOptions")
print("=" * 60)

default_opts = onnxruntime.SessionOptions()
default_results = benchmark("default", default_opts)

for name, r in default_results.items():
    print(f"  {name}: {r['avg_ms']:7.1f} ms  RTF={r['rtf']:.4f}  audio={r['audio_sec']:.2f}s")

DEFAULT SessionOptions
  short  (20 phonemes):   112.4 ms  RTF=0.1861  audio=0.60s
  medium (100 phonemes):   265.2 ms  RTF=0.1785  audio=1.49s
  long   (300 phonemes):   804.5 ms  RTF=0.2179  audio=3.69s


In [9]:
print("=" * 60)
print("OPTIMIZED SessionOptions")
print("=" * 60)

opt_opts = create_session_options()
print(f"  intra_op_num_threads: {opt_opts.intra_op_num_threads}")
print(f"  inter_op_num_threads: {opt_opts.inter_op_num_threads}")
print()

opt_results = benchmark("optimized", opt_opts)

for name, r in opt_results.items():
    print(f"  {name}: {r['avg_ms']:7.1f} ms  RTF={r['rtf']:.4f}  audio={r['audio_sec']:.2f}s")

OPTIMIZED SessionOptions
  intra_op_num_threads: 1
  inter_op_num_threads: 1

  short  (20 phonemes):   109.9 ms  RTF=0.1821  audio=0.60s
  medium (100 phonemes):   342.0 ms  RTF=0.2301  audio=1.49s
  long   (300 phonemes):   697.7 ms  RTF=0.1890  audio=3.69s


## 4. 比較結果

In [10]:
print("=" * 60)
print("DEFAULT vs OPTIMIZED")
print("=" * 60)
print(f"{'Test':<25s} | {'DEFAULT (ms)':>12s} | {'OPTIMIZED (ms)':>14s} | {'Speedup':>8s} | {'RTF':>6s}")
print("-" * 75)

for name in default_results:
    d = default_results[name]["avg_ms"]
    o = opt_results[name]["avg_ms"]
    rtf = opt_results[name]["rtf"]
    pct = (d - o) / d * 100 if d > 0 else 0
    rt_factor = 1.0 / rtf if rtf > 0 else float("inf")
    print(f"  {name:<23s} | {d:12.1f} | {o:14.1f} | {pct:+7.1f}% | {rtf:.4f} ({rt_factor:.0f}x RT)")

DEFAULT vs OPTIMIZED
Test                      | DEFAULT (ms) | OPTIMIZED (ms) |  Speedup |    RTF
---------------------------------------------------------------------------
  short  (20 phonemes)    |        112.4 |          109.9 |    +2.2% | 0.1821 (5x RT)
  medium (100 phonemes)   |        265.2 |          342.0 |   -28.9% | 0.2301 (4x RT)
  long   (300 phonemes)   |        804.5 |          697.7 |   +13.3% | 0.1890 (5x RT)


In [11]:
result_json = {
    "environment": {
        "platform": platform.platform(),
        "cpu": platform.processor(),
        "logical_cores": os.cpu_count(),
        "onnxruntime_version": onnxruntime.__version__,
        "model_size_mb": round(model_size_mb, 1),
    },
    "settings": {"warmup_runs": WARMUP, "benchmark_runs": RUNS},
    "default": {k: v for k, v in default_results.items()},
    "optimized": {k: v for k, v in opt_results.items()},
}

print(json.dumps(result_json, indent=2, ensure_ascii=False))

{
  "environment": {
    "platform": "Linux-6.6.113+-x86_64-with-glibc2.35",
    "cpu": "x86_64",
    "logical_cores": 2,
    "onnxruntime_version": "1.24.4",
    "model_size_mb": 37.4
  },
  "settings": {
    "warmup_runs": 3,
    "benchmark_runs": 10
  },
  "default": {
    "short  (20 phonemes)": {
      "avg_ms": 112.37029370004166,
      "rtf": 0.18613018149683883,
      "audio_sec": 0.603718820861678
    },
    "medium (100 phonemes)": {
      "avg_ms": 265.2227579999817,
      "rtf": 0.17847173504332264,
      "audio_sec": 1.486077097505669
    },
    "long   (300 phonemes)": {
      "avg_ms": 804.5428599000388,
      "rtf": 0.21791679025152141,
      "audio_sec": 3.6919727891156464
    }
  },
  "optimized": {
    "short  (20 phonemes)": {
      "avg_ms": 109.93241299997862,
      "rtf": 0.1820920753192254,
      "audio_sec": 0.603718820861678
    },
    "medium (100 phonemes)": {
      "avg_ms": 341.95471190000717,
      "rtf": 0.2301056334654284,
      "audio_sec": 1.486077097

## 5. 参考: 他の TTS との比較

KittenTTS issue #40 / sherpa-onnx (Google Colab CPU, single thread):

| TTS Engine | RTF | Model Size | Notes |
|---|---|---|---|
| **MatchaTTS + Vocos** | **0.163** | 123 MB | Fastest (2-model) |
| **Piper Plus (FP16)** | **0.192** | **38 MB** | Best speed/size ratio |
| Piper FP32 | 0.276 | 75 MB | |
| Piper INT8 | 0.523 | 22 MB | Quantization hurts |
| KittenTTS FP16 | 0.693 | 23 MB | English only |
| Kokoro FP32 | 1.880 | 330 MB | Below realtime |
| Kokoro INT8 | 3.564 | 128 MB | INT8 even slower |

> Source: https://github.com/KittenML/KittenTTS/issues/40